<a href="https://colab.research.google.com/github/sabascodes/Logistics-project/blob/main/delay_model_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Barakah Logistics – Shipment Delay Prediction
Two models: a classifier (will the shipment be delayed?) and a regressor (how many hours late?). The `predict_delay()` function at the end is what the AI agent will call.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, mean_absolute_error, r2_score)

In [2]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# accept either Excel or CSV
if file_name.endswith(".xlsx"):
    pd.read_excel(file_name).to_csv("delay_model_dataset.csv", index=False)
    DATA_PATH = "delay_model_dataset.csv"
else:
    DATA_PATH = file_name

Saving delay_model_dataset.csv to delay_model_dataset.csv


## 1. Settings

In [3]:
MODEL_PATH = "delay_model.joblib"
RANDOM_STATE = 42

CATEGORICAL = ["origin_city", "destination_city", "transportation_mode",
               "season", "weather_condition", "traffic_level", "priority_level"]
NUMERIC = ["distance_km", "shipment_weight_kg", "order_lead_time_hours",
           "carrier_rating", "vehicle_capacity_utilization",
           "planned_transit_hours", "month", "day_of_week"]
FEATURES = CATEGORICAL + NUMERIC

In [4]:
def add_date_features(df):
    # turn shipment_date into month and day_of_week
    df = df.copy()
    dates = pd.to_datetime(df["shipment_date"])
    df["month"] = dates.dt.month
    df["day_of_week"] = dates.dt.dayofweek
    return df

def make_preprocessor():
    # text columns -> one-hot (0/1 columns), number columns -> scaled
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", StandardScaler(), NUMERIC),
    ])

## 2. Load data and split

In [5]:
df = add_date_features(pd.read_csv(DATA_PATH))
df.head()

,shipment_id,origin_city,destination_city,transportation_mode,distance_km,shipment_date,season,weather_condition,traffic_level,priority_level,shipment_weight_kg,order_lead_time_hours,carrier_rating,vehicle_capacity_utilization,planned_transit_hours,actual_transit_hours,delay_hours,is_delayed,month,day_of_week
0,SHP00001,Riyadh,Hail,air,627,2024-11-15,Regular,Extreme Heat,Low,Standard,697.9,29.1,3.5,0.57,5.65,6.58,0.93,0,11,4
1,SHP00002,Riyadh,Mecca,air,867,2024-08-29,Regular,Extreme Heat,Medium,Urgent,551.1,9.8,4.0,0.44,6.41,4.91,-1.50,0,8,3
2,SHP00003,Riyadh,Medina,truck,838,2023-08-08,Regular,Clear,High,Standard,726.6,15.0,2.9,0.70,14.93,21.62,6.69,1,8,1
3,SHP00004,Riyadh,Dammam,train,400,2024-01-31,Regular,Clear,Low,Standard,1148.6,19.7,4.3,0.69,6.80,6.74,-0.06,0,1,2
4,SHP00005,Jeddah,Hail,truck,930,2025-01-10,Regular,Clear,Low,Standard,1662.7,11.6,3.2,0.85,16.63,16.90,0.27,0,1,4


In [6]:
X = df[FEATURES]
y_class = df["is_delayed"]    # 0 = on time, 1 = delayed
y_hours = df["delay_hours"]   # can be negative = arrived early

# 80% to learn from, 20% kept hidden to test honestly
# stratify keeps the same % of delayed shipments in train and test
X_tr, X_te, yc_tr, yc_te, yh_tr, yh_te = train_test_split(
    X, y_class, y_hours, test_size=0.2, stratify=y_class, random_state=RANDOM_STATE
)
print(f"Rows: {len(df)} | Train rows: {len(X_tr)} | Test rows: {len(X_te)}")
print(f"Delayed shipments: {y_class.mean():.1%}")

Rows: 6000 | Train rows: 4800 | Test rows: 1200
Delayed shipments: 47.7%


## 3. Delay classifier: baseline vs Random Forest
Logistic Regression is a simple baseline. Random Forest should beat it to be worth using.

In [7]:
candidates = {
    "Logistic Regression (baseline)": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            random_state=RANDOM_STATE, n_jobs=-1),
}

fitted = {}
for model_name, model in candidates.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
    pipe.fit(X_tr, yc_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    fitted[model_name] = pipe
    print(f"{model_name:32s} accuracy={accuracy_score(yc_te, pred):.3f}  "
          f"precision={precision_score(yc_te, pred):.3f}  "
          f"recall={recall_score(yc_te, pred):.3f}  "
          f"F1={f1_score(yc_te, pred):.3f}  "
          f"AUC={roc_auc_score(yc_te, proba):.3f}")

Logistic Regression (baseline)   accuracy=0.801  precision=0.786  recall=0.801  F1=0.793  AUC=0.876
Random Forest                    accuracy=0.833  precision=0.835  recall=0.812  F1=0.823  AUC=0.906


In [8]:
clf = fitted["Random Forest"]
proba = clf.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(yc_te, pred).ravel()
print("Confusion matrix (Random Forest):")
print(f"  correct on-time: {tn} | false alarms: {fp}")
print(f"  missed delays:   {fn} | caught delays: {tp}")

Confusion matrix (Random Forest):
  correct on-time: 535 | false alarms: 92
  missed delays:   108 | caught delays: 465


## 4. Overfitting check
If train accuracy is much higher than test accuracy, the model is overfitting. Cross-validation tests the model on 5 different splits instead of trusting just one.

In [9]:
train_acc = accuracy_score(yc_tr, clf.predict(X_tr))
test_acc  = accuracy_score(yc_te, pred)
print(f"Train accuracy: {train_acc:.2%}")
print(f"Test accuracy:  {test_acc:.2%}")
print(f"Gap:            {train_acc - test_acc:.2%}  (small gap = not overfitting)\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_pipe = Pipeline([("prep", make_preprocessor()),
                    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                     random_state=RANDOM_STATE, n_jobs=-1))])
cv_auc = cross_val_score(cv_pipe, X, y_class, cv=cv, scoring="roc_auc")
print(f"Test AUC:          {roc_auc_score(yc_te, proba):.3f}")
print(f"5-fold CV AUC:     {cv_auc.mean():.3f} (+/- {cv_auc.std():.3f})  (close to test AUC = reliable)")

Train accuracy: 89.83%
Test accuracy:  83.33%
Gap:            6.50%  (small gap = not overfitting)

Test AUC:          0.906
5-fold CV AUC:     0.899 (+/- 0.006)  (close to test AUC = reliable)


## 5. Delay-hours regressor
Predicts how many hours late. We compare it to just guessing the average every time.

In [10]:
reg = Pipeline([("prep", make_preprocessor()),
                ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                random_state=RANDOM_STATE, n_jobs=-1))])
reg.fit(X_tr, yh_tr)
h_pred = reg.predict(X_te)

baseline_mae = mean_absolute_error(yh_te, np.full(len(yh_te), yh_tr.mean()))
print(f"MAE:          {mean_absolute_error(yh_te, h_pred):.2f} hours")
print(f"Baseline MAE: {baseline_mae:.2f} hours  (guessing the average every time)")
print(f"R^2:          {r2_score(yh_te, h_pred):.3f}")

MAE:          1.32 hours
Baseline MAE: 2.44 hours  (guessing the average every time)
R^2:          0.682


## 6. What matters most?

In [11]:
names = clf.named_steps["prep"].get_feature_names_out()
imp = pd.Series(clf.named_steps["model"].feature_importances_, index=names)
imp.index = [n.split("__", 1)[1] for n in imp.index]   # remove the "cat__" / "num__" prefix
print("Top 10 factors driving delay (Random Forest):")
print(imp.sort_values(ascending=False).head(10).round(3).to_string())

Top 10 factors driving delay (Random Forest):
weather_condition_Clear         0.134
weather_condition_Sandstorm     0.092
transportation_mode_truck       0.087
order_lead_time_hours           0.087
planned_transit_hours           0.081
traffic_level_Low               0.075
vehicle_capacity_utilization    0.069
carrier_rating                  0.053
traffic_level_High              0.045
shipment_weight_kg              0.041


## 7. Sanity check on the new city: Hail

In [12]:
test_df = X_te.copy()
test_df["y"] = yc_te
test_df["pred"] = pred
hail = test_df[test_df["destination_city"] == "Hail"]
print(f"Hail test rows: {len(hail)} | Hail accuracy: {(hail['y'] == hail['pred']).mean():.2%}")
print(f"Overall accuracy: {test_acc:.2%}  (similar = model handles Hail well)")

Hail test rows: 264 | Hail accuracy: 84.09%
Overall accuracy: 83.33%  (similar = model handles Hail well)


## 8. Final model
Now that we've tested it, retrain on ALL the data and save it for the agent.

In [13]:
clf.fit(X, y_class)
reg.fit(X, y_hours)
joblib.dump({"classifier": clf, "regressor": reg, "features": FEATURES}, MODEL_PATH)
print(f"Saved final models to {MODEL_PATH}")

# optional: download the saved model to your device
# files.download(MODEL_PATH)

Saved final models to delay_model.joblib


## 9. Predictions
A helper that takes one shipment and returns a result the AI agent can read directly. `shipment_date` is given as `'YYYY-MM-DD'`.

In [14]:
_bundle = None

def predict_delay(shipment):
    global _bundle
    if _bundle is None:
        _bundle = joblib.load(MODEL_PATH)   # load the saved model once

    row = add_date_features(pd.DataFrame([shipment]))[_bundle["features"]]
    prob = float(_bundle["classifier"].predict_proba(row)[0, 1])
    hours = max(float(_bundle["regressor"].predict(row)[0]), 0)   # early arrival counts as 0
    planned = float(shipment["planned_transit_hours"])

    if prob < 0.35:
        risk = "Low"
    elif prob < 0.65:
        risk = "Medium"
    else:
        risk = "High"

    return {
        "delay_probability": round(prob, 3),
        "risk_level": risk,
        "expected_delay_hours": round(hours, 2),
        "planned_transit_hours": planned,
        "expected_transit_hours": round(planned + hours, 2),
    }

In [15]:
# same shipment, bad conditions vs good conditions
bad = {
    "origin_city": "Riyadh", "destination_city": "Hail",
    "transportation_mode": "truck", "distance_km": 627,
    "shipment_date": "2025-03-10", "season": "Ramadan",
    "weather_condition": "Sandstorm", "traffic_level": "High",
    "priority_level": "Urgent", "shipment_weight_kg": 800,
    "order_lead_time_hours": 12, "carrier_rating": 3.2,
    "vehicle_capacity_utilization": 0.9, "planned_transit_hours": 11.5,
}
good = {**bad, "weather_condition": "Clear", "traffic_level": "Low", "carrier_rating": 4.8}

print("Riyadh -> Hail, truck, sandstorm, high traffic:")
print(predict_delay(bad))
print("\nRiyadh -> Hail, truck, clear weather, low traffic:")
print(predict_delay(good))

Riyadh -> Hail, truck, sandstorm, high traffic:
{'delay_probability': 0.976, 'risk_level': 'High', 'expected_delay_hours': 11.93, 'planned_transit_hours': 11.5, 'expected_transit_hours': 23.43}

Riyadh -> Hail, truck, clear weather, low traffic:
{'delay_probability': 0.168, 'risk_level': 'Low', 'expected_delay_hours': 0.26, 'planned_transit_hours': 11.5, 'expected_transit_hours': 11.76}
